# SIFLOW · run_5 · Dream-7B setup (SIFLOW-D)

Downloads Dream-7B (~14GB), verifies a masked forward, and tokenizes data in Dream's tokenizer. Trains live in run_6 (no cache needed). ~1h.

**Runtime:** comfortably under one Colab session.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
# === Hugging Face login ===
# Required for the gated DiffusionGemma weights; recommended for Dream too.
# Get a token at https://huggingface.co/settings/tokens (read scope).
from huggingface_hub import login
login()

> If `tokenizer.mask_token_id` is unset for Dream, set `teacher.mask_token` in `siflow/config/dream.yaml`. If a raw forward lacks `.logits`, set `teacher.auto_class`.

In [ ]:
import torch
from siflow.teacher import DreamTeacher
teacher = DreamTeacher(dtype=torch.bfloat16)
print("vocab", teacher.vocab_size, "hidden", teacher.hidden_dim, "mask_id", teacher.mask_index)
print("argmax:", teacher.logits(torch.full((1,16), teacher.mask_index, device=teacher.device)).argmax(-1)[0][:8].tolist())

In [ ]:
from siflow.data import build_token_chunks
tokz = teacher.tokenizer
print("dream chunks:",
      build_token_chunks(tokz, 256, 60_000, f"{BASE}/data/dream_256.npy", dataset="Skylion007/openwebtext", split="train"),
      build_token_chunks(tokz, 256, 2_000, f"{BASE}/data/dream_val.npy", dataset="Skylion007/openwebtext", split="train", skip_seqs=60_000))

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_5_dream_data.zip', include=['data/dream_256.npy', 'data/dream_val.npy'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)